# Klasifikasi teks MULTI-LABEL: ujaran kebencian & bahasa kasar di Twitter Indonesia

**Butuh:** `train` (kolom `Tweet` + 12 kolom label 0/1), `test` (kolom `id` + `Tweet`).
**Menghasilkan:** `submission.csv` (kolom `id` + 12 kolom label 0/1), probabilitas tiap model di
`outputs/`, laporan F1 per label.

Dataset: Ibrohim & Budi (2019), *Multi-label Hate Speech and Abusive Language Detection in
Indonesian Twitter*. `data.csv` aslinya satu file tanpa pembagian, jadi dibagi **80/20 ala Kaggle**
(`siapkan_data.py`): `hs_train.csv` (label ada), `hs_test.csv` (tanpa label),
`hs_test_labels.csv` (kunci jawaban, hanya untuk cek akhir), `hs_sample_submission.csv`.
Isi tweet tidak diubah; tweet yang sama persis dijaga tetap di satu sisi supaya tidak bocor.

## 12 label dan strukturnya (terverifikasi di data)

| kelompok | kolom | aturan |
|---|---|---|
| induk | `HS` (ujaran kebencian), `Abusive` (kata kasar) | independen satu sama lain |
| target | `HS_Individual`, `HS_Group` | kalau `HS=1`: tepat **satu** yang bernilai 1 |
| tingkat | `HS_Weak`, `HS_Moderate`, `HS_Strong` | kalau `HS=1`: tepat **satu** yang bernilai 1 |
| kategori | `HS_Religion`, `HS_Race`, `HS_Physical`, `HS_Gender`, `HS_Other` | kalau `HS=1`: minimal satu (kadang 2–3) |
| semua `HS_*` | | kalau `HS=0`: semuanya **0** |

Struktur ini dipakai dua kali: (1) model sub-label dilatih khusus pada tweet `HS=1`, lalu dikali
`P(HS)`; (2) hasil akhir didekode agar patuh aturan di atas.

**Metrik:** F1 macro atas 12 label (rata-rata F1 kelas positif tiap kolom). Label jarang
(`HS_Physical`, `HS_Gender`) sama bobotnya dengan `Abusive`, jadi ambang per label (sel 2) penting.

## Cara menempel ke notebook-mu

| sel | isi | wajib? |
|---|---|---|
| 0 | data (pola sama dengan template: `train`, `test`, `test_labels`) | **ya** |
| 1 | konfigurasi, split 80/20 tetap (dipakai SEMUA model) | **ya** |
| 2 | fungsi bantu: F1, ambang per label, dekode hierarki, `fit_predict_proba` | **ya** |
| 3 | lihat data singkat | opsional |
| 4 | TF-IDF + leksikon kata kasar + LogReg (dari `tfidf.ipynb`) | **ya** (baseline, ±10 detik) |
| 5 | vektor kalimat MiniLM + LogReg (dari `embedding.ipynb`) | disarankan |
| 6a–6b | fine-tune IndoBERT multi-label (dari `finetune_bert.ipynb`, butuh GPU) | paling akurat |
| 7 | gabung model yang ada → bobot, ambang, dekode → `submission.csv` | **ya** |
| 8 | cek ke kunci jawaban (hanya demo) | opsional |

Tiap model menyimpan `outputs/val_proba_<nama>.npy` dan `outputs/test_proba_<nama>.npy`. Sel 7
otomatis memakai semua yang ada, jadi 4, 5, 6 boleh dijalankan terpisah (mis. 6 di Colab/GPU).

## 0. Data (pola sama dengan template; isi tidak diubah)
`test_labels` = kunci jawaban (hanya ada di demo). Kolom teks tetap bernama `Tweet`.

In [14]:
import pandas as pd

DEMO_FOLDER = "."   # folder data_latihan (di sebelah repo)

train = pd.read_csv(f"hs_train.csv")
test = pd.read_csv(f"hs_test.csv")
test_labels = pd.read_csv(f"hs_test_labels.csv")
sample_submission = pd.read_csv(f"hs_sample_submission.csv")
print(train.shape, test.shape, sample_submission.shape)

(10533, 14) (2636, 2) (2636, 13)


## 1. Konfigurasi + split 80/20 tetap
**Menghasilkan:** `LABELS`, `Y`, `index_train`, `index_val`, indeks hierarki (`SUB_IDX`, dst).
Split dibuat sekali di sini dan dipakai semua model, sehingga probabilitas validasi mereka
sejajar baris-per-baris dan bisa digabung. Stratifikasi memakai kombinasi 12 label (kombinasi
< 20 baris digabung jadi satu kelompok) dan tweet kembar dijaga satu sisi.

In [15]:
import os
import re

import numpy as np
import torch
from sklearn.model_selection import StratifiedGroupKFold

TEXT = "Tweet"                          # UBAH
LABELS = [c for c in train.columns if c not in ("id", TEXT)]
assert LABELS == [c for c in test_labels.columns if c != "id"] == [c for c in sample_submission.columns if c != "id"]
RESOURCE_FOLDER = f"{DEMO_FOLDER}"   # abusive.csv + new_kamusalay.csv
PRETRAINED_FOLDER = r"C:\INAH-MINE\GEMASTIK-XIX\pretrained"       # UBAH
RUN_EMBEDDING = True                   # UBAH: False kalau belum ada model minilm
RUN_BERT = True                        # UBAH: False kalau tidak ada GPU
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs("outputs", exist_ok=True)

train[TEXT] = train[TEXT].fillna("").astype(str)
test[TEXT] = test[TEXT].fillna("").astype(str)

# struktur hierarki label
HS_IDX = LABELS.index("HS")
SUB_IDX = [i for i, name in enumerate(LABELS) if name.startswith("HS_")]
TYPE_IDX = [LABELS.index(n) for n in ("HS_Individual", "HS_Group")]
LEVEL_IDX = [LABELS.index(n) for n in ("HS_Weak", "HS_Moderate", "HS_Strong")]
CATEGORY_IDX = [LABELS.index(n) for n in ("HS_Religion", "HS_Race", "HS_Physical", "HS_Gender", "HS_Other")]

Y = train[LABELS].to_numpy().astype(int)
combo = pd.Series(["".join(map(str, row)) for row in Y])
strat_key = np.where(combo.map(combo.value_counts()) >= 20, combo, "jarang")
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
index_train, index_val = next(splitter.split(Y, strat_key, groups=train[TEXT]))
print(len(LABELS), "label |", len(index_train), "train |", len(index_val), "val | device:", DEVICE)
print(pd.DataFrame({"train": Y[index_train].mean(axis=0), "val": Y[index_val].mean(axis=0)},
                   index=LABELS).round(3).T.to_string())

12 label | 8426 train | 2107 val | device: cuda
          HS  Abusive  HS_Individual  HS_Group  HS_Religion  HS_Race  HS_Physical  HS_Gender  HS_Other  HS_Weak  HS_Moderate  HS_Strong
train  0.422    0.384          0.272     0.150        0.061    0.042        0.025      0.024     0.284    0.258        0.130      0.035
val    0.422    0.380          0.271     0.151        0.059    0.043        0.025      0.021     0.284    0.255        0.129      0.037


## 2. Fungsi bantu
- `f1_per_label`, `macro_f1`: metrik (vektor, cepat).
- `tune_thresholds`: cari ambang terbaik **per label** di validasi (default 0,5 tidak optimal untuk label jarang).
- `decode_hierarchy`: ubah probabilitas jadi 0/1 yang patuh aturan hierarki.
- `fit_predict_proba`: latih satu model untuk 12 label; `conditional=True` menambah model
  sub-label yang hanya dilatih pada tweet `HS=1`, lalu `P(sub) = P(HS) × P(sub | HS)`.

In [16]:
GRID = np.arange(0.10, 0.91, 0.05)


def f1_per_label(y_true, y_bin):
    tp = (y_true * y_bin).sum(axis=0)
    fp = ((1 - y_true) * y_bin).sum(axis=0)
    fn = (y_true * (1 - y_bin)).sum(axis=0)
    return 2 * tp / np.maximum(2 * tp + fp + fn, 1)


def macro_f1(y_true, y_bin):
    return f1_per_label(y_true, y_bin).mean()


def tune_thresholds(y_true, proba):
    scores = np.array([f1_per_label(y_true, (proba >= g).astype(int)) for g in GRID])   # (ambang, label)
    return GRID[scores.argmax(axis=0)]


def decode_hierarchy(proba, thresholds):
    binary = (proba >= thresholds).astype(int)
    is_hs = binary[:, HS_IDX] == 1
    binary[np.ix_(~is_hs, SUB_IDX)] = 0                      # HS=0 -> semua sub-label 0
    for group in (TYPE_IDX, LEVEL_IDX):                      # tepat satu per kelompok
        chosen = np.eye(len(group), dtype=int)[proba[:, group].argmax(axis=1)]
        binary[np.ix_(is_hs, group)] = chosen[is_hs]
    no_category = is_hs & (binary[:, CATEGORY_IDX].sum(axis=1) == 0)   # minimal satu kategori
    best = np.array(CATEGORY_IDX)[proba[:, CATEGORY_IDX].argmax(axis=1)]
    binary[no_category, best[no_category]] = 1
    return binary


def fit_predict_proba(make_estimator, X_fit, Y_fit, X_pred, conditional=True):
    proba = make_estimator().fit(X_fit, Y_fit).predict_proba(X_pred)
    if conditional:
        is_hs = Y_fit[:, HS_IDX] == 1
        sub_model = make_estimator().fit(X_fit[is_hs], Y_fit[is_hs][:, SUB_IDX])
        proba[:, SUB_IDX] = proba[:, [HS_IDX]] * sub_model.predict_proba(X_pred)
    return proba


def save_proba(name, val_proba, test_proba):
    np.save(f"outputs/val_proba_{name}.npy", val_proba)
    np.save(f"outputs/test_proba_{name}.npy", test_proba)


def report(y_true, y_bin):
    f1 = f1_per_label(y_true, y_bin)
    return pd.DataFrame({"jumlah_positif": y_true.sum(axis=0), "diprediksi": y_bin.sum(axis=0),
                         "F1": f1.round(4)}, index=LABELS)


print("fungsi siap; 12 label, sub-label HS_*:", len(SUB_IDX))

fungsi siap; 12 label, sub-label HS_*: 10


## 3. Lihat data singkat
Cek struktur hierarki, tweet kembar, dan panjang teks. Teks mentah mengandung `USER`/`URL`
(pengganti akun/tautan), sisa byte emoji `\xf0\x9f...`, dan `\n` literal.

In [17]:
print("HS=0 tetapi ada sub-label:", int(((Y[:, HS_IDX] == 0) & (Y[:, SUB_IDX].sum(axis=1) > 0)).sum()))
is_hs = Y[:, HS_IDX] == 1
print("HS=1: jumlah target (harus 1):", set(Y[is_hs][:, TYPE_IDX].sum(axis=1)),
      "| jumlah tingkat (harus 1):", set(Y[is_hs][:, LEVEL_IDX].sum(axis=1)),
      "| jumlah kategori:", sorted(set(Y[is_hs][:, CATEGORY_IDX].sum(axis=1))))
print("tweet kembar di train:", int(train.duplicated(TEXT).sum()),
      "| panjang kata median/95%:", np.percentile(train[TEXT].str.split().str.len(), [50, 95]))
print("mengandung USER:", round(train[TEXT].str.contains("USER").mean(), 3),
      "| mengandung URL:", round(train[TEXT].str.contains("URL").mean(), 3))
print(train[[TEXT] + LABELS[:2]].sample(5, random_state=1).to_string(max_colwidth=80))

HS=0 tetapi ada sub-label: 0
HS=1: jumlah target (harus 1): {np.int64(1)} | jumlah tingkat (harus 1): {np.int64(1)} | jumlah kategori: [np.int64(1), np.int64(2), np.int64(3)]
tweet kembar di train: 117 | panjang kata median/95%: [15. 38.]
mengandung USER: 0.583 | mengandung URL: 0.063
                                                                                Tweet  HS  Abusive
1005                                      Sekjen PBB keknya bagian dari Saracen.. :-)   1        0
6968                                             USER USER kemana aja sinting WKWKWK'   0        1
1049  Dulu para pahlawan berjuang susah payah, korban jiwa raga utk usir penjajah....   1        0
8143                                         USER USER USER Celeng nyebut celeng? :)'   1        1
5822  Barusan, aku nonton film Monster. Film ttg Aileen Wournos. OMG sedih:(\nSela...   0        0


## 4. TF-IDF + leksikon kata kasar + LogisticRegression (dari `tfidf.ipynb`)
Fitur: TF-IDF kata 1–2 + potongan huruf 2–5 (tahan salah ketik/gaul) + 6 angka dari
`abusive.csv` (jumlah kata kasar, panjang, dst; setelah normalisasi `new_kamusalay.csv`).
Teks **mentah** (huruf kecil) dipakai untuk TF-IDF: pada uji kami, pembersihan penuh tidak
menambah skor karena n-gram huruf sudah menyerap variasi tulisan.
Satu `LogisticRegression` per label (`OneVsRestClassifier`), `class_weight="balanced"`.

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import FeatureUnion, make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

abusive_words = set(pd.read_csv(f"{RESOURCE_FOLDER}/abusive.csv")["ABUSIVE"].str.lower())
slang = pd.read_csv(f"{RESOURCE_FOLDER}/new_kamusalay.csv", header=None, names=["alay", "baku"],
                    encoding="latin-1")
SLANG_TO_FORMAL = dict(zip(slang["alay"], slang["baku"]))


def normalize_words(text):
    # huruf kecil, buang sisa byte emoji / \n literal / tanda baca, rapikan huruf berulang
    text = re.sub(r"\\x[0-9a-f]{2}", " ", str(text).lower()).replace("\\n", " ")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"(.)\1{2,}", r"\1", text)
    return [SLANG_TO_FORMAL.get(word, word) for word in text.split()]


def lexicon_features(texts):
    rows = []
    for text in texts:
        words = normalize_words(text)
        n_abusive = sum(word in abusive_words for word in words)
        rows.append([n_abusive, n_abusive > 0, n_abusive / max(1, len(words)), len(words),
                     str(text).count("USER"), str(text).isupper()])
    return np.array(rows, dtype=float)


def make_tfidf_model(C=2):
    features = FeatureUnion([
        ("kata", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("huruf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True)),
        ("leksikon", make_pipeline(FunctionTransformer(lexicon_features), StandardScaler()))])
    classifier = LogisticRegression(C=C, max_iter=3000, class_weight="balanced")
    return make_pipeline(features, OneVsRestClassifier(classifier, n_jobs=-1))


texts = train[TEXT].to_numpy()
Y_val = Y[index_val]
best_C, best_score = None, -1.0
for C in (1, 2, 5):                                           # UBAH
    proba = fit_predict_proba(lambda: make_tfidf_model(C), texts[index_train], Y[index_train],
                              texts[index_val])
    score = macro_f1(Y_val, (proba >= tune_thresholds(Y_val, proba)).astype(int))
    print(f"C={C}: F1 macro val (ambang per label): {score:.4f}")
    if score > best_score:
        best_C, best_score, val_proba_tfidf = C, score, proba
print("C terbaik:", best_C)

C=1: F1 macro val (ambang per label): 0.7217
C=2: F1 macro val (ambang per label): 0.7262
C=5: F1 macro val (ambang per label): 0.7274
C terbaik: 5


In [19]:
# latih ulang di SEMUA train -> probabilitas test; simpan untuk sel 7
test_proba_tfidf = fit_predict_proba(lambda: make_tfidf_model(best_C), texts, Y, test[TEXT].to_numpy())
save_proba("tfidf", val_proba_tfidf, test_proba_tfidf)
thr = tune_thresholds(Y_val, val_proba_tfidf)
print("F1 macro val TF-IDF:", round(macro_f1(Y_val, decode_hierarchy(val_proba_tfidf, thr)), 4), "(ambang + hierarki)")
print(report(Y_val, decode_hierarchy(val_proba_tfidf, thr)).T.to_string())

F1 macro val TF-IDF: 0.7163 (ambang + hierarki)
                     HS   Abusive  HS_Individual  HS_Group  HS_Religion  HS_Race  HS_Physical  HS_Gender  HS_Other   HS_Weak  HS_Moderate  HS_Strong
jumlah_positif  889.000  800.0000       571.0000  318.0000     125.0000  90.0000      52.0000    45.0000  598.0000  538.0000     272.0000    79.0000
diprediksi      876.000  811.0000       563.0000  313.0000     134.0000  88.0000      51.0000    46.0000  636.0000  525.0000     270.0000    81.0000
F1                0.851    0.9174         0.7178    0.6688       0.7027   0.7528       0.4466     0.6374    0.7925    0.6943       0.6015     0.8125


## 5. Vektor kalimat MiniLM + LogisticRegression (dari `embedding.ipynb`)
Menangkap **makna** (sinonim/parafrase) sehingga melengkapi TF-IDF yang menangkap kata kunci.
Teks dirapikan ringan lewat `neural_text` (huruf kecil, `USER` berulang jadi satu `@user`,
sisa byte emoji dibuang), karena model bahasa tidak butuh `USER USER USER ...`.
Vektor di-cache di `outputs/` supaya tidak dihitung ulang. Lewati kalau `RUN_EMBEDDING=False`.

In [20]:
def neural_text(text):
    text = re.sub(r"\\x[0-9a-f]{2}", " ", str(text).lower()).replace("\\n", " ")
    text = re.sub(r"(?:\buser\b\s*)+", "@user ", text)
    text = re.sub(r"\burl\b", "tautan", text)
    text = re.sub(r"\brt\b", " ", text)
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    return re.sub(r"\s+", " ", text).strip(" '\"")


train["neural"] = train[TEXT].map(neural_text)
test["neural"] = test[TEXT].map(neural_text)
print(train[["neural"]].head(3).to_string(max_colwidth=90))

if RUN_EMBEDDING:
    from sentence_transformers import SentenceTransformer

    if os.path.exists("outputs/emb_train.npy") and os.path.exists("outputs/emb_test.npy"):
        X_emb, X_emb_test = np.load("outputs/emb_train.npy"), np.load("outputs/emb_test.npy")
    else:
        encoder = SentenceTransformer(f"{PRETRAINED_FOLDER}/minilm", device=DEVICE)   # UBAH: minilm_en
        X_emb = encoder.encode(train["neural"].tolist(), batch_size=64, normalize_embeddings=True,
                               show_progress_bar=True)
        X_emb_test = encoder.encode(test["neural"].tolist(), batch_size=64, normalize_embeddings=True)
        np.save("outputs/emb_train.npy", X_emb)
        np.save("outputs/emb_test.npy", X_emb_test)
    print(X_emb.shape, X_emb_test.shape)

                                                                                      neural
0  - disaat semua cowok berusaha melacak perhatian gue. loe lantas remehkan perhatian yg ...
1  @user : @user siapa yang telat ngasih tau elu?edan sarap gue bergaul dengan cigax jifl...
2  41. kadang aku berfikir, kenapa aku tetap percaya pada tuhan padahal aku selalu jatuh ...


Batches: 100%|██████████| 165/165 [00:09<00:00, 17.23it/s]


(10533, 384) (2636, 384)


In [21]:
if RUN_EMBEDDING:
    def make_embedding_model(C=2):
        classifier = LogisticRegression(C=C, max_iter=3000, class_weight="balanced")
        return OneVsRestClassifier(classifier, n_jobs=-1)

    best_C_emb, best_score = None, -1.0
    for C in (1, 2, 5):                                       # UBAH
        proba = fit_predict_proba(lambda: make_embedding_model(C), X_emb[index_train], Y[index_train],
                                  X_emb[index_val])
        score = macro_f1(Y_val, (proba >= tune_thresholds(Y_val, proba)).astype(int))
        print(f"C={C}: F1 macro val (ambang per label): {score:.4f}")
        if score > best_score:
            best_C_emb, best_score, val_proba_emb = C, score, proba
    test_proba_emb = fit_predict_proba(lambda: make_embedding_model(best_C_emb), X_emb, Y, X_emb_test)
    save_proba("embedding", val_proba_emb, test_proba_emb)
    print("C terbaik:", best_C_emb, "| tersimpan: outputs/*_proba_embedding.npy")

C=1: F1 macro val (ambang per label): 0.5149
C=2: F1 macro val (ambang per label): 0.5201
C=5: F1 macro val (ambang per label): 0.5268
C terbaik: 5 | tersimpan: outputs/*_proba_embedding.npy


## 6a. Fine-tune IndoBERT multi-label: dataset + fungsi latih (dari `finetune_bert.ipynb`)
Perbedaan dari versi 1-label: keluaran 12 logit dengan `BCEWithLogitsLoss` + `sigmoid`
(bukan softmax), `labels` berupa vektor float 12 angka, dan `pos_weight` (akar rasio
negatif/positif, maks 5) agar label jarang tidak diabaikan. Pilih epoch dengan F1 macro val
terbaik (ambang 0,5); ambang per label disetel di sel 7. Pengaturan lain tetap dari template
(lr 3e-5, batch 16, warmup 10% + turun linear, bf16, berhenti dini).
Hasil tiap seed di-cache; jalankan ulang tidak melatih ulang kecuali `FORCE_RETRAIN=True`.
Tweet pendek (median 15 kata, maks 52), jadi `MAX_LENGTH=128` aman; "CUDA out of memory":
turunkan `BATCH_SIZE` ke 8.

In [22]:
# [BERT] definisi: lewati sel 6a-6b kalau tidak ada GPU (RUN_BERT=False)
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          get_linear_schedule_with_warmup)

MODEL_NAME = "indobert"            # UBAH
MAX_LENGTH = 128                   # UBAH
BATCH_SIZE = 16                    # UBAH
EPOCHS = 4                         # UBAH: 3-5 biasanya cukup
LEARNING_RATE = 3e-5               # UBAH: 1e-5 .. 5e-5
PATIENCE = 2                       # berhenti kalau F1 val tidak naik 2 epoch berturut-turut
SEEDS = [42]                       # UBAH: [42, 7, 2024] -> hasil dirata-rata (lebih stabil)
USE_POS_WEIGHT = True
USE_BF16 = DEVICE == "cuda"        # UBAH: False kalau GPU lama / error bf16
FORCE_RETRAIN = False

if RUN_BERT:
    tokenizer = AutoTokenizer.from_pretrained(f"{PRETRAINED_FOLDER}/{MODEL_NAME}")
    token_counts = [len(tokenizer(t)["input_ids"]) for t in train["neural"].head(2000)]
    print("token per teks, persentil 50/95/99:", np.percentile(token_counts, [50, 95, 99]))


class TextDataset(Dataset):
    def __init__(self, texts, targets):
        self.tokens = tokenizer(list(texts), truncation=True, max_length=MAX_LENGTH)
        self.targets = np.asarray(targets, dtype=np.float32)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, i):
        return {"input_ids": self.tokens["input_ids"][i],
                "attention_mask": self.tokens["attention_mask"][i],
                "labels": self.targets[i].tolist()}


def run_epoch(network, loader, loss_function, optimizer=None, scheduler=None):
    training = optimizer is not None
    network.train(training)
    total_loss, probabilities = 0.0, []
    for batch in tqdm(loader, leave=False):
        batch = batch.to(DEVICE)
        targets = batch.pop("labels")
        with torch.set_grad_enabled(training):
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=USE_BF16):
                logits = network(**batch).logits.float()
            loss = loss_function(logits, targets)
        if training:
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(network.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
        total_loss += loss.item() * len(targets)
        probabilities.append(torch.sigmoid(logits.detach()).cpu().numpy())
    return total_loss / len(loader.dataset), np.concatenate(probabilities)


def train_one_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    network = AutoModelForSequenceClassification.from_pretrained(
        f"{PRETRAINED_FOLDER}/{MODEL_NAME}", num_labels=len(LABELS),
        problem_type="multi_label_classification").to(DEVICE)
    positives = Y[index_train].sum(axis=0)
    ratio = np.sqrt((len(index_train) - positives) / np.maximum(positives, 1))
    pos_weight = torch.tensor(np.clip(ratio, 1.0, 5.0), dtype=torch.float).to(DEVICE)
    loss_function = nn.BCEWithLogitsLoss(pos_weight=pos_weight if USE_POS_WEIGHT else None)
    optimizer = torch.optim.AdamW(network.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
    total_steps = EPOCHS * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
    best_f1, waited = -1.0, 0
    for epoch in range(EPOCHS):
        train_loss, _ = run_epoch(network, train_loader, loss_function, optimizer, scheduler)
        val_loss, val_proba = run_epoch(network, val_loader, loss_function)
        val_f1 = macro_f1(Y[index_val], (val_proba >= 0.5).astype(int))
        print(f"seed {seed} epoch {epoch + 1}: train {train_loss:.3f} | val {val_loss:.3f} | F1 {val_f1:.4f}")
        if val_f1 > best_f1:
            best_f1, waited = val_f1, 0
            torch.save(network.state_dict(), f"outputs/bert_seed{seed}.pt")
        else:
            waited += 1
        if waited >= PATIENCE:
            print("berhenti dini")
            break
    network.load_state_dict(torch.load(f"outputs/bert_seed{seed}.pt", map_location=DEVICE))
    _, val_proba = run_epoch(network, val_loader, loss_function)
    _, test_proba = run_epoch(network, test_loader, loss_function)
    del network, optimizer
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return val_proba, test_proba

token per teks, persentil 50/95/99: [24. 56. 69.]


## 6b. Jalankan (tiap seed disimpan; `bert` = rata-rata seed)

In [23]:
# [BERT] jalankan
if RUN_BERT:
    collate = DataCollatorWithPadding(tokenizer)
    train_data = TextDataset(train["neural"].to_numpy()[index_train], Y[index_train])
    val_data = TextDataset(train["neural"].to_numpy()[index_val], Y[index_val])
    test_data = TextDataset(test["neural"].to_numpy(), np.zeros((len(test), len(LABELS))))   # label pengisi
    val_loader = DataLoader(val_data, batch_size=BATCH_SIZE * 2, collate_fn=collate)
    test_loader = DataLoader(test_data, batch_size=BATCH_SIZE * 2, collate_fn=collate)

    seed_val, seed_test = [], []
    for seed in SEEDS:
        files = (f"outputs/val_proba_bert_seed{seed}.npy", f"outputs/test_proba_bert_seed{seed}.npy")
        if all(os.path.exists(f) for f in files) and not FORCE_RETRAIN:
            val_proba, test_proba = np.load(files[0]), np.load(files[1])
            print(f"seed {seed}: pakai hasil tersimpan")
        else:
            val_proba, test_proba = train_one_seed(seed)
            np.save(files[0], val_proba)
            np.save(files[1], test_proba)
        print(f"seed {seed}: F1 macro val (ambang per label):",
              round(macro_f1(Y_val, (val_proba >= tune_thresholds(Y_val, val_proba)).astype(int)), 4))
        seed_val.append(val_proba)
        seed_test.append(test_proba)
    save_proba("bert", np.mean(seed_val, axis=0), np.mean(seed_test, axis=0))
    print("tersimpan: outputs/*_proba_bert.npy (rata-rata", len(SEEDS), "seed)")

[transformers] You passed `num_labels=12` which is incompatible to the `id2label` map of length `5`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13537.91it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: C:\INAH-MINE\GEMASTIK-XIX\pretrained/indobert
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


seed 42 epoch 1: train 0.459 | val 0.304 | F1 0.6889


seed 42 epoch 2: train 0.262 | val 0.272 | F1 0.7274


seed 42 epoch 3: train 0.159 | val 0.259 | F1 0.7569


seed 42 epoch 4: train 0.097 | val 0.275 | F1 0.7624


seed 42: F1 macro val (ambang per label): 0.7696
tersimpan: outputs/*_proba_bert.npy (rata-rata 1 seed)


## 7. Gabung semua model yang ada → `submission.csv`
1. Muat `outputs/*_proba_<nama>.npy` yang tersedia (tfidf, embedding, bert).
2. Cari bobot rata-rata probabilitas (kelipatan 0,1) dengan F1 macro val terbaik (ambang per label disetel untuk tiap kandidat).
3. Setel ambang per label pada probabilitas gabungan, lalu dekode: bandingkan **dengan** dan **tanpa** aturan hierarki, pakai yang lebih baik di val.
4. Terapkan bobot, ambang, dan dekode yang sama ke test → `submission.csv` (`id` + 12 kolom 0/1, urutan kolom sama dengan `hs_sample_submission.csv`).

In [24]:
import itertools

names = [n for n in ("tfidf", "embedding", "bert")
         if os.path.exists(f"outputs/val_proba_{n}.npy") and os.path.exists(f"outputs/test_proba_{n}.npy")]
val_probas = {n: np.load(f"outputs/val_proba_{n}.npy") for n in names}
test_probas = {n: np.load(f"outputs/test_proba_{n}.npy") for n in names}
print("model tersedia:", names)


def weight_grid(n_models, step=0.1):
    total = int(round(1 / step))
    for parts in itertools.product(range(total + 1), repeat=n_models):
        if sum(parts) == total:
            yield np.array(parts) / total


def blend(probas, weights):
    return sum(w * probas[n] for w, n in zip(weights, names))


best_weights, best_score = None, -1.0
for weights in weight_grid(len(names)):
    mixed = blend(val_probas, weights)
    score = macro_f1(Y_val, (mixed >= tune_thresholds(Y_val, mixed)).astype(int))
    if score > best_score:
        best_weights, best_score = weights, score
print("bobot terbaik:", dict(zip(names, best_weights)), "| F1 macro val:", round(best_score, 4))

mixed_val = blend(val_probas, best_weights)
thresholds = tune_thresholds(Y_val, mixed_val)
plain_score = macro_f1(Y_val, (mixed_val >= thresholds).astype(int))
hier_score = macro_f1(Y_val, decode_hierarchy(mixed_val, thresholds))
USE_HIERARCHY = hier_score >= plain_score
print(f"val tanpa hierarki {plain_score:.4f} | dengan hierarki {hier_score:.4f} -> pakai hierarki: {USE_HIERARCHY}")
print(pd.Series(thresholds, index=LABELS).round(2).to_string())

model tersedia: ['tfidf', 'embedding', 'bert']
bobot terbaik: {'tfidf': np.float64(0.4), 'embedding': np.float64(0.1), 'bert': np.float64(0.5)} | F1 macro val: 0.7758
val tanpa hierarki 0.7758 | dengan hierarki 0.7657 -> pakai hierarki: False
HS               0.45
Abusive          0.45
HS_Individual    0.35
HS_Group         0.35
HS_Religion      0.35
HS_Race          0.40
HS_Physical      0.40
HS_Gender        0.35
HS_Other         0.50
HS_Weak          0.35
HS_Moderate      0.40
HS_Strong        0.40


In [25]:
import pandas as pd

mixed_test = blend(test_probas, best_weights)
if USE_HIERARCHY:
    test_binary = decode_hierarchy(mixed_test, thresholds)
else:
    test_binary = (mixed_test >= thresholds).astype(int)

submission = pd.DataFrame(test_binary, columns=LABELS)
submission.insert(0, "id", test["id"].to_numpy())                           # UBAH
assert list(submission.columns) == list(sample_submission.columns)
submission.to_csv("submission.csv", index=False)
pd.DataFrame(mixed_test, columns=LABELS).assign(id=test["id"].to_numpy()).to_csv(
    "outputs/test_proba_ensemble.csv", index=False)
print(submission.shape)
print(submission[LABELS].mean().round(3).to_string())

(2636, 13)
HS               0.432
Abusive          0.403
HS_Individual    0.313
HS_Group         0.181
HS_Religion      0.078
HS_Race          0.051
HS_Physical      0.023
HS_Gender        0.023
HS_Other         0.281
HS_Weak          0.302
HS_Moderate      0.156
HS_Strong        0.043


## 8. Cek ke kunci jawaban (hanya demo; saat lomba tidak ada)
Ambang dan bobot ditentukan hanya dari validasi, jadi skor test ini jujur. Tabel kedua
membandingkan tiap model sendirian (ambang per label dari val) dengan gabungan.

In [26]:
Y_test = test_labels[LABELS].to_numpy().astype(int)
print("urutan id sama:", (test_labels["id"].to_numpy() == submission["id"].to_numpy()).all())
print("F1 macro test (gabungan):", round(macro_f1(Y_test, submission[LABELS].to_numpy()), 4),
      "| exact match:", round((Y_test == submission[LABELS].to_numpy()).all(axis=1).mean(), 4))
print(report(Y_test, submission[LABELS].to_numpy()).to_string())

rows = []
for n in names:
    thr_n = tune_thresholds(Y_val, val_probas[n])
    rows.append({"model": n, "F1 val": macro_f1(Y_val, (val_probas[n] >= thr_n).astype(int)),
                 "F1 test": macro_f1(Y_test, (test_probas[n] >= thr_n).astype(int))})
rows.append({"model": "gabungan", "F1 val": hier_score if USE_HIERARCHY else plain_score,
             "F1 test": macro_f1(Y_test, submission[LABELS].to_numpy())})
print(pd.DataFrame(rows).round(4).to_string(index=False))

urutan id sama: True
F1 macro test (gabungan): 0.7519 | exact match: 0.6882
               jumlah_positif  diprediksi      F1
HS                       1114        1140  0.8767
Abusive                  1009        1062  0.9174
HS_Individual             709         825  0.7575
HS_Group                  405         477  0.7302
HS_Religion               157         205  0.7403
HS_Race                   120         135  0.8157
HS_Physical                61          61  0.5574
HS_Gender                  59          60  0.6218
HS_Other                  747         740  0.8151
HS_Weak                   673         796  0.7447
HS_Moderate               339         412  0.6684
HS_Strong                 102         114  0.7778
    model  F1 val  F1 test
    tfidf  0.7274   0.7326
embedding  0.5268   0.5057
     bert  0.7696   0.7332
 gabungan  0.7758   0.7519
